# Stanza で多言語の形態素・依存構造解析をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_parsing/stanza.ipynb)

NLP クックブック 18-07 ／ 構文解析

日本語・英語以外も含めて、70 以上の言語を同じ API で解析したい（Universal Dependencies 準拠）。

In [1]:
!pip install -q stanza

In [2]:
import stanza

stanza.download("ja", verbose=False)
nlp = stanza.Pipeline("ja", processors="tokenize,pos,lemma,depparse", verbose=False)
doc = nlp("太郎は花子に本を渡した。")
for sent in doc.sentences:
    for w in sent.words:
        head = sent.words[w.head - 1].text if w.head > 0 else "ROOT"
        print(f"{w.id:2} {w.text:4} {w.lemma:4} {w.upos:6} {w.deprel:8} → {head}")

 1 太郎   太郎   PROPN  nsubj    → 渡し
 2 は    は    ADP    case     → 太郎
 3 花子   花子   PROPN  obl      → 渡し
 4 に    に    ADP    case     → 花子
 5 本    本    NOUN   obj      → 渡し
 6 を    を    ADP    case     → 本
 7 渡し   渡す   VERB   root     → ROOT
 8 た    た    AUX    aux      → 渡し
 9 。    。    PUNCT  punct    → 渡し


In [3]:
# 言語を変えても同じコードで動く
for lang, text in [("de", "Der Hund jagt die Katze."), ("ko", "나는 학교에 갑니다.")]:
    stanza.download(lang, verbose=False)
    p = stanza.Pipeline(lang, processors="tokenize,pos,lemma,depparse", verbose=False)
    print(lang, [(w.text, w.upos, w.deprel) for s in p(text).sentences for w in s.words])

de [('Der', 'DET', 'det'), ('Hund', 'NOUN', 'nsubj'), ('jagt', 'VERB', 'root'), ('die', 'DET', 'det'), ('Katze', 'NOUN', 'obj'), ('.', 'PUNCT', 'punct')]
ko [('나는', 'PRON', 'dislocated'), ('학교에', 'ADV', 'obl'), ('갑니다', 'VERB', 'root'), ('.', 'PUNCT', 'punct')]


- `w.head` は 1 始まりの単語番号で、0 が ROOT。
- GPU があれば自動で使う。CPU に固定するなら `stanza.Pipeline(..., use_gpu=False)`。